# Notebook 01.1 — Aquisição IBGE: malhas territoriais e dados socioeconômicos

**Projeto:** Acessibilidade Geográfica às UBS de Teresina — roteiro computacional AE2SFCA  
**Programa:** MAPEPROF — Mestrado Profissional em Planejamento Urbano e Regional / IFPI  
**Autor:** Felipe Ramos Dantas  
**Orientador:** Prof. Dr. Antonio Joaquim da Silva  
**Coorientador:** Prof. Dr. Reurysson Chagas de Sousa Morais  
**Repositório:** https://github.com/felipedantas-pi/ae2sfca-ubs  
**Última atualização:** 2026-06-08

## Objetivo

Baixar e pré-processar os dados geoespaciais e tabulares do **IBGE** necessários ao pipeline AE2SFCA. O processo inclui a delimitação da área de estudo (zona urbana + buffer de 5 km restrito ao limite municipal), a aquisição das geometrias censitárias e a extração e limpeza dos microdados do Censo 2022.

## Saídas

Gravadas em `dados/externos/ibge/`:

| Arquivo | Descrição |
|---|---|
| `teresina_municipio.parquet` | Limite municipal de Teresina |
| `teresina_bairros.parquet` | Malha de bairros oficiais |
| `teresina_zonaUrbana_utm.parquet` | Perímetro da zona urbana (bairros dissolvidos) |
| `teresina_zonaUrbana_buffer5kClip_utm.parquet` | Área de estudo: ZU + buffer 5 km $\cap$ limite municipal |
| `teresina_setoresCensitariosUrbanos.parquet` | Setores censitários restritos à situação urbana |
| `teresina_gradeEstatistica_utm.parquet` | Grade estatística intersecionada com o limite municipal |

Gravado em `dados/intermediarios/01_malha/`:

| Arquivo | Descrição |
|---|---|
| `teresina_setoresCensitarios_DadosCompletos.parquet` | GeoDataFrame mestre fundido com dados socioeconômicos (renda, cor/raça, demografia) |

## Pré-requisitos

- Conexão de internet estável (downloads do IBGE podem chegar a centenas de MB).
- Notebook executado uma única vez por release dos dados IBGE.
- Tempo médio: ~10–15 min.

---

In [1]:
# ── 1. IMPORTAÇÕES E CONFIGURAÇÃO GLOBAL ────────────────────────────────────
import pandas as pd
import geopandas as gpd
import requests
import zipfile
import io

# Caminhos e CRS centralizados (ver src/mapeprof/config.py)
from mapeprof.config import (
    EXT_IBGE,        # dados/externos/ibge   — onde gravar as malhas
    INT_MALHA,       # dados/intermediarios/01_malha  — onde gravar o master
    CRS_METRICO,     # EPSG:31983 — SIRGAS 2000 / UTM 23S
    CRS_GEOGRAFICO,  # EPSG:4674  — SIRGAS 2000 geográfico
    criar_diretorios,
)

criar_diretorios()
print("Dependências e diretórios prontos.")

Dependências e diretórios prontos.


In [2]:
# ── 2. DOWNLOAD E PRÉ-PROCESSAMENTO DAS MALHAS TERRITORIAIS ─────────────────

print("🌍 Baixando malhas territoriais do IBGE (Piauí)...")
url_pi_municipios    = "https://geoftp.ibge.gov.br/organizacao_do_territorio/malhas_territoriais/malhas_municipais/municipio_2025/UFs/PI/PI_Municipios_2025.zip"
url_pi_bairros = "https://geoftp.ibge.gov.br/organizacao_do_territorio/malhas_territoriais/malhas_de_setores_censitarios__divisoes_intramunicipais/censo_2022/bairros/shp/UF/PI_bairros_CD2022.zip"

gdf_pi_mun        = gpd.read_file(f"zip+{url_pi_municipios}")
gdf_pi_bairro = gpd.read_file(f"zip+{url_pi_bairros}")

print("✂️ Filtrando Teresina e estruturando a Zona Urbana...")
gdf_teresina_mun         = gdf_pi_mun.query("NM_MUN == 'Teresina'").copy()
gdf_teresina_bairros = gdf_pi_bairro.query("NM_MUN == 'Teresina'").copy()

# Dissolve os limites internos dos bairros para gerar o polígono contínuo da Zona Urbana
gdf_zonaUrbana = gdf_teresina_bairros.dissolve()[['CD_MUN', 'NM_MUN', 'geometry']]

# Construção da área de estudo: reprojeção métrica → buffer 5 km → reprojeção geográfica → clip
gdf_zonaUrbana_5km = gdf_zonaUrbana.to_crs(CRS_METRICO)
gdf_zonaUrbana_5km['geometry'] = gdf_zonaUrbana_5km.buffer(distance=5000, cap_style='flat', join_style='bevel')
gdf_zonaUrbana_5km = gdf_zonaUrbana_5km.to_crs(CRS_GEOGRAFICO)

# Mantém a zona urbana expandida estritamente dentro do município de Teresina
gdf_zonaUrbana_5km_clip = gpd.clip(gdf_zonaUrbana_5km, gdf_teresina_mun)

# Exportação em GeoParquet, todos reprojetados para o CRS métrico
print("💾 Exportando arquivos territoriais...")
gdf_teresina_mun.to_crs(CRS_METRICO).to_parquet(EXT_IBGE / "teresina_municipio.parquet", index=False)
gdf_teresina_bairros.to_crs(CRS_METRICO).to_parquet(EXT_IBGE / "teresina_bairros.parquet", index=False)
gdf_zonaUrbana.to_crs(CRS_METRICO).to_parquet(EXT_IBGE / "teresina_zonaUrbana_utm.parquet", index=False)
gdf_zonaUrbana_5km_clip.to_crs(CRS_METRICO).to_parquet(EXT_IBGE / "teresina_zonaUrbana_buffer5kClip_utm.parquet", index=False)

print("✅ Malhas territoriais processadas.")

🌍 Baixando malhas territoriais do IBGE (Piauí)...
✂️ Filtrando Teresina e estruturando a Zona Urbana...
💾 Exportando arquivos territoriais...
✅ Malhas territoriais processadas.


In [3]:
# ── 3. DOWNLOAD E CRUZAMENTO ESPACIAL (SETORES E GRADE) ─────────────────────

print("📊 Baixando setores censitários e grade estatística do IBGE...")
url_pi_setores = "https://geoftp.ibge.gov.br/organizacao_do_territorio/malhas_territoriais/malhas_de_setores_censitarios__divisoes_intramunicipais/censo_2022/setores/shp/UF/PI_setores_CD2022.zip"
url_grade   = "https://geoftp.ibge.gov.br/recortes_para_fins_estatisticos/grade_estatistica/censo_2022/grade_estatistica/grade_id66.zip"

# Apenas setores classificados como "Urbana" em Teresina
gdf_teresina_scUrbano = gpd.read_file(f"zip+{url_pi_setores}").query(
    "NM_MUN == 'Teresina' and SITUACAO == 'Urbana'"
)

# Grade estatística nacional
gdf_gradeEstatistica = gpd.read_file(f"zip+{url_grade}")

print("✂️ Cruzando grade estatística com os limites de Teresina (overlay)...")
gdf_grade_intersec = gpd.overlay(
    gdf_gradeEstatistica,
    gdf_teresina_mun[['CD_MUN', 'NM_MUN', 'geometry']],
    how='intersection'
)

print("💾 Exportando arquivos censitários...")
gdf_teresina_scUrbano.to_crs(CRS_METRICO).to_parquet(EXT_IBGE / "teresina_setoresCensitariosUrbanos.parquet", index=False)
gdf_grade_intersec.to_crs(CRS_METRICO).to_parquet(EXT_IBGE / "teresina_gradeEstatistica_utm.parquet", index=False)

print("✅ Geometrias censitárias processadas.")

📊 Baixando setores censitários e grade estatística do IBGE...
✂️ Cruzando grade estatística com os limites de Teresina (overlay)...
💾 Exportando arquivos censitários...
✅ Geometrias censitárias processadas.


In [4]:
# ── 4. DECLARAÇÃO DE FUNÇÕES E URLS DO PIPELINE ETL ──────────────────────────

URL_BASE_SETORES = "https://ftp.ibge.gov.br/Censos/Censo_Demografico_2022/Agregados_por_Setores_Censitarios/Agregados_por_Setor_csv/"
URL_BASE_RENDA   = "https://ftp.ibge.gov.br/Censos/Censo_Demografico_2022/Agregados_por_Setores_Censitarios_Rendimento_do_Responsavel/"

# Dicionário de configuração das variáveis extraídas
TABELAS_CENSO = {
    'corRaca': {
        'url': f"{URL_BASE_SETORES}Agregados_por_setores_cor_ou_raca_BR.zip",
        'colunas': ['CD_SETOR', 'V01317', 'V01318', 'V01319', 'V01320', 'V01321']
    },
    'demog': {
        'url': f"{URL_BASE_SETORES}Agregados_por_setores_demografia_BR.zip",
        'colunas': ['CD_SETOR', 'V01006', 'V01031', 'V01032', 'V01033', 'V01034', 'V01035', 'V01036', 'V01037', 'V01038', 'V01039', 'V01040', 'V01041']
    },
    'renda': {
        'url': f"{URL_BASE_RENDA}Agregados_por_setores_renda_responsavel_BR_20260508_csv.zip",
        'colunas': ['CD_SETOR', 'V06003', 'V06004', 'V06005', 'V06006']
    }
}

def extrair_e_limpar_censo(url, colunas_alvo, sufixo):
    """
    Realiza o download em memória (io.BytesIO), filtra colunas, resolve
    inconsistências de uppercase/lowercase e limpa os dados censurados do IBGE.
    """
    print(f"  -> Processando grupo: {sufixo.upper()}...")
    resposta = requests.get(url)

    # Trava o código de forma limpa e mostra a URL exata se o IBGE não retornar sucesso (Código 200)
    if resposta.status_code != 200:
        raise ValueError(f"\n❌ Erro de Conexão: O arquivo não foi encontrado no servidor.\nO IBGE retornou o código HTTP {resposta.status_code}.\nVerifique no navegador se o nome ou a pasta do arquivo mudou.\nLink tentado: {url}")
    
    with zipfile.ZipFile(io.BytesIO(resposta.content), 'r') as z:
        nome_csv = [n for n in z.namelist() if n.endswith('.csv')][0]
        with z.open(nome_csv) as f_csv:
            # Leitura de cabeçalho para mapeamento de case sensitivity
            amostra = pd.read_csv(f_csv, sep=';', nrows=0)
            colunas_arquivo = amostra.columns.tolist()
            mapa_colunas = {col.upper(): col for col in colunas_arquivo}
            
            # Ajusta as colunas alvo para o formato exato que existe no arquivo
            colunas_reais_para_ler = [mapa_colunas[c] for c in colunas_alvo]
            
            # Leitura da tabela completa com filtro de uso e tipagem estrita
            f_csv.seek(0)
            df = pd.read_csv(
                f_csv, 
                sep=';', 
                usecols=colunas_reais_para_ler,
                dtype={mapa_colunas['CD_SETOR']: str},
                low_memory=False
            )
            
    # Padroniza todas as colunas para caixa alta (resolve o 'CD_setor' do IBGE)
    df.rename(str.upper, axis='columns', inplace=True)
    
    # Renomeia adicionando o sufixo (blindando a chave primária CD_SETOR)
    dict_renomear = {c: f"{c}_{sufixo}" for c in colunas_alvo if c != 'CD_SETOR'}
    df.rename(columns=dict_renomear, inplace=True)
    
    # Limpeza estatística: Converte 'X' e '-' para 0 numérico (int/float)
    cols_limpar = list(dict_renomear.values())
    for col in cols_limpar:
        df[col] = pd.to_numeric(df[col].replace(['X', '-'], 0), errors='coerce').fillna(0)
        
    return df

In [5]:
# ── 5. EXECUÇÃO DO PIPELINE ETL E MERGE ESPACIAL ────────────────────────────

print("📥 Iniciando extração remota dos dados tabulares (Censo 2022)...")

# Base geográfica que receberá as junções
gdf_master = gdf_teresina_scUrbano.copy()

for chave, config in TABELAS_CENSO.items():
    df_temp = extrair_e_limpar_censo(config['url'], config['colunas'], chave)
    # how='left' garante que apenas os setores de Teresina permaneçam
    gdf_master = gdf_master.merge(df_temp, on='CD_SETOR', how='left')

caminho_final = INT_MALHA / "teresina_setoresCensitarios_DadosCompletos.parquet"
gdf_master.to_crs(CRS_METRICO).to_parquet(caminho_final, index=False)

print("\n🎉 Pipeline concluído.")
print(f"GeoDataFrame mestre: {len(gdf_master):,} setores × {len(gdf_master.columns)} atributos.")
print(f"Arquivo salvo em: {caminho_final}")

📥 Iniciando extração remota dos dados tabulares (Censo 2022)...
  -> Processando grupo: CORRACA...
  -> Processando grupo: DEMOG...
  -> Processando grupo: RENDA...

🎉 Pipeline concluído.
GeoDataFrame mestre: 1,403 setores × 51 atributos.
Arquivo salvo em: C:\Users\felipe\workspace_pcksa\ae2sfca_ubs\dados\intermediarios\01_malha\teresina_setoresCensitarios_DadosCompletos.parquet
